# VS Code + AgentCore Gateway: Serverless OAuth Proxy

## Overview

This notebook deploys a **serverless OAuth proxy** using API Gateway + Lambda,
eliminating the need for developers to run local proxy and callback servers.

### Architecture

![](./generated-diagrams/image.png)

### What This Deploys

1. **API Gateway** - Public endpoint for VS Code to connect to
2. **MCP Proxy Lambda** - OAuth metadata, callback interception, token proxying, MCP forwarding
3. **3LO Callback Lambda** - Outbound OAuth callbacks, CompleteResourceTokenAuth
4. **Cognito User Pool** - JWT tokens for inbound authentication
5. **DynamoDB Table** - Table to store temporary codes
5. **AgentCore Gateway** - MCP server with Confluence target

## Step 1: Setup

In [ ]:
# Install dependencies
!pip3 install -r requirements.txt --quiet

Deploy the CDK stack:

```bash
cd cdk
npm install
cdk deploy
```

then copy the output in the cell below replacing the content.

In [ ]:
import boto3

In [59]:
output = """
FigmaMCP.ApiEndpoint = https://q934p5v250.execute-api.eu-west-1.amazonaws.com/
FigmaMCP.CallbackLambdaName = FigmaMCP-McpCallbackLambdaAE4B86C1-Qwp0yKLqujAR
FigmaMCP.CognitoDomain = agentcore-figma
FigmaMCP.CognitoDomainUrl = https://agentcore-figma.auth.eu-west-1.amazoncognito.com
FigmaMCP.DiscoveryUrl = https://cognito-idp.eu-west-1.amazonaws.com/eu-west-1_AZSTXTv7c/.well-known/openid-configuration
FigmaMCP.Gateway = agentcore-figma-gateway-jkndmt24t1
FigmaMCP.M2MClientId = 
FigmaMCP.ProxyLambdaName = FigmaMCP-McpProxyLambda5036A849-K6twBip7vxaY
FigmaMCP.UserPoolArn = arn:aws:cognito-idp:eu-west-1:416075262792:userpool/eu-west-1_AZSTXTv7c
FigmaMCP.UserPoolId = eu-west-1_AZSTXTv7c
FigmaMCP.VSCodeClientId = 
FigmaMCP.VSCodeMcpConfig = {
  "servers": {
    "agentcore-confluence": {
      "type": "http",
      "url": "https://q934p5v250.execute-api.eu-west-1.amazonaws.com/mcp",
      "headers": {
        "MCP-Protocol-Version": "2025-11-25"
      }
    }
  }
}
"""

In [ ]:
config = {}
for el in output.split("\n"):
    if "=" in el:
        key, value = el.split("=", 1)
        config[key.strip().replace("FigmaMCP.", "")] = value.strip()

## Step 1: Create Cognito User

In [ ]:
COGNITO_USERNAME = "vscode-user@example.com"
COGNITO_PASSWORD = "TempPassword123!"

cognito = boto3.client("cognito-idp")
user_pool_id = config["UserPoolId"]
try:
    cognito.admin_create_user(
        UserPoolId=user_pool_id,
        Username=COGNITO_USERNAME,
        TemporaryPassword=COGNITO_PASSWORD,
        MessageAction='SUPPRESS',
        UserAttributes=[{'Name': 'email', 'Value': f'{COGNITO_USERNAME}'},
                       {'Name': 'email_verified', 'Value': 'true'}]
    )
    cognito.admin_set_user_password(
        UserPoolId=user_pool_id, Username=COGNITO_USERNAME,
        Password=COGNITO_PASSWORD, Permanent=True
    )
    print(f"✓ User created: {COGNITO_USERNAME}")
except cognito.exceptions.UsernameExistsException:
    print(f"✓ User exists: {COGNITO_USERNAME}")

## Step 3: Create Figma Credential Provider

In [ ]:
import boto3

ac = boto3.client('bedrock-agentcore-control')
acr = boto3.client('bedrock-agentcore')

In [ ]:
credential_provider_name = "figma-dcr"
figma_provider = {}

In [ ]:
# ac.delete_oauth2_credential_provider(name=credential_provider_name)

In [ ]:
figma_provider = ac.create_oauth2_credential_provider(
    name=credential_provider_name, 
    credentialProviderVendor='CustomOauth2', 
    oauth2ProviderConfigInput={
    'customOauth2ProviderConfig': {
        'clientId': 'aaa',
        'clientSecret': 'bbb',
        'oauthDiscovery': {
            'discoveryUrl': 'https://api.figma.com/.well-known/oauth-authorization-server',
        }
    }})

In [ ]:
credential_provider_arn = figma_provider.get('credentialProviderArn', None)
agentcore_callback_url = figma_provider.get('callbackUrl', None)


In [ ]:
import requests
import json

registration = requests.post(
    'https://api.figma.com/v1/oauth/mcp/register', 
    headers={'content-type': 'application/json', 'accept': 'application=/json'}, 
    data=json.dumps({
      "redirect_uris": [
        agentcore_callback_url,
        ],
        "response_types": ["code"],
        "grant_types": ["authorization_code", "refresh_token"],
      "client_name": "Visual Studio Code",
      "client_uri": "https://code.visualstudio.com"
     })
)
registration.text

In [ ]:
figma_provider =  ac.update_oauth2_credential_provider(
    name=credential_provider_name, credentialProviderVendor='CustomOauth2', oauth2ProviderConfigInput={
    'customOauth2ProviderConfig': {
        'clientId': registration.json()['client_id'],
        'clientSecret': registration.json()['client_secret'],
        'oauthDiscovery': {
            'discoveryUrl': 'https://api.figma.com/.well-known/oauth-authorization-server',
        }
    }})

In [ ]:
figma_provider

In [ ]:
figma_provider=ac.get_oauth2_credential_provider(name=credential_provider_name)

## Step 4: Create Figma Target

In [ ]:
GATEWAY_ID = config["Gateway"]

In [ ]:
ac.list_gateway_targets(gatewayIdentifier=GATEWAY_ID)['items']

In [ ]:
target_response=ac.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId='FZQVE7DAW5')

In [ ]:
# ac.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId='FZQVE7DAW5')

In [ ]:
api_endpoint = config["ApiEndpoint"]
DEFAULT_RETURN_URL = f"{api_endpoint}oauth2/callback"
config_target = {
    "name": "new",
    "credentialProviderConfigurations":[{
                             'credentialProviderType':'OAUTH',
                             'credentialProvider': {'oauthCredentialProvider': {
                                 'providerArn': figma_provider['credentialProviderArn'] , 
                                 'scopes': ['mcp:connect'],
                                 'defaultReturnUrl': DEFAULT_RETURN_URL,
                                 'grantType': 'AUTHORIZATION_CODE'}}
                             }],
    "targetConfiguration":{'mcp':{'mcpServer': {'endpoint':'https://mcp.figma.com/mcp'}}}
}

In [ ]:
target_response = ac.update_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId='FZQVE7DAW5', **config_target)

In [ ]:
import json

target_response = ac.create_gateway_target(gatewayIdentifier=GATEWAY_ID, name='new', 
                         **config)

target_id = target_response["targetId"]
print(f"✓ Confluence target: {target_id}")

In [ ]:
target_id = target_response["targetId"]
print(f"✓ Confluence target: {target_id}")

In [ ]:
auth_url = target_response['authorizationData']['oauth2']['authorizationUrl']
user_id = target_response['authorizationData']['oauth2']['userId']
sessionUri = auth_url.split('=')[1].replace('%3A', ':')

In [ ]:
print("Click on the following URL to authorize the target")
print(auth_url)

In [ ]:
acr.complete_resource_token_auth(sessionUri=sessionUri, 
                                 userIdentifier={'userId':user_id} )

In [ ]:
print(f'https://console.aws.amazon.com/bedrock-agentcore/toolsAndGateways/{GATEWAY_ID}')

## Step 6: VS Code Configuration

No local servers needed! Just configure VS Code to point to the API Gateway.

In [ ]:
print(json.dumps({
  "servers": {
    "figma-agentcore": {
      "type": "http",
      "url": config["ApiEndpoint"]+"mcp",
      "headers": {
        "MCP-Protocol-Version": "2025-11-25"
      }
    }
  }
}, indent=2))

----

## Cleanup (Optional)

Run this cell to delete all resources created by this notebook.

In [ ]:
client = boto3.client('bedrock-agentcore-control')

def cleanup():
    """Delete all resources created by this notebook."""
    print("Cleaning up resources...")
    # Delete Gateway target and gateway
    targets = client.list_gateway_targets(gatewayIdentifier=config["Gateway"])['items']
    try:
        for t in targets:
            print(f" Deleting target {t['targetId']} for gateway {config["Gateway"]}", end='')
            client.delete_gateway_target(gatewayIdentifier=config["Gateway"], targetId=t['targetId'])
            print(" done")

    except: pass
    
    # Delete credential provider
    try:
        client.delete_oauth2_credential_provider(name=credential_provider_name)
        print(f"✓ Deleted Credential Provider")
    except: pass

# Uncomment to run cleanup:
cleanup()

To remove the remaining resources run:

```bash
cd cdk
cdk destroy
```